# Fetching Hidden and Cell States of an LSTM Cell

While building an LSTM network, we can fetch the output value of the previous timestamp from the hidden layer using the **return_sequences** argument passed in the LSTM method. This way we not only have the output of the final timestamp but also the subsequent timestamp outputs. It is not always beneficial to get the hidden state output every time, only for a few cases, this may be helpful like machine translation.

We will use one LSTM cell along with one hidden layer and try to get the output for 5 timestamps:

In [1]:
# Installing necessary packages
%pip install keras numpy

# Importing necessary methods
from keras.models import Model
from keras.layers import Input, LSTM
import numpy as np


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


2025-06-16 19:07:53.973713: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-16 19:07:54.251760: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-16 19:07:54.271786: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750100874.288950   46087 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750100874.297299   46087 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1750100874.323649   46087 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:
# Defining five inputs
inputs = np.array([0.2, 0.3, 0.4, 0.5, 0.6]).reshape((1, 5, 1))

# Defining LSTM network
np.random.seed(42)
feed = Input(shape=(5, 1))
lstm = LSTM(1, return_sequences=True)(feed)
model = Model(inputs=feed, outputs=lstm)

# Predictions
print('Outputs from each five timestamps')
model.predict(inputs)

2025-06-16 19:08:04.428272: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Outputs from each five timestamps
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 193ms/step


array([[[0.01788708],
        [0.03577578],
        [0.05241303],
        [0.06714352],
        [0.0797287 ]]], dtype=float32)

Not only output (hidden state) but you can also fetch the cell state using the return_state argument. Modify the above code with these two lines and observe the change:

In [3]:
lstm, state_h, state_c = LSTM(1, return_sequences=True, return_state=True)(feed)
model = Model(inputs=feed, outputs=(lstm, state_h, state_c))

# Dropouts

Usually, to apply dropout in a neural network you may use the Dropout method from the Keras such as:

In [4]:
# model.add(Dropout(0.3))

However, if you try to use this mechanism of dropout for RNNs (like LSTM/GRU) it can interfere with the timestamps (can even drop them) unless you rely on its argument **noise_shape**. Since we know that a recurrent layer takes 2 inputs at a timestamp, your input and the internal input (can be just the output of the previous state or including the cell state, depending upon the architecture used). Therefore, it is not always necessary that the output and/or cell state from the previous state may match the dimension of the current input. So, Keras provides two different dropouts to handle this situation.

   - dropout: Current input
   - recurrent_dropout: Recurrent state (previous output and/or cell state)

Both of these arguments take a value between 0 and 1 to drop a fraction of units for the linear transformation of the respective input.